In [1]:
import random
import torch
import os
import re

import pandas as pd
import polars as pl
import numpy as np

import sys
sys.path.append('../')
import compcor.corpus_metrics as corpus_metrics
from compcor.utils import Corpus
from compcor.text_tokenizer_embedder import STTokenizerEmbedder

from collections import defaultdict

/home/isabel/anaconda3/envs/localSyntheticData/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/283 [00:00<?, ?it/s]

In [2]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    pl.set_random_seed(random_state)
    pd.core.common.random_state(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

In [3]:
# ------------------ Metric Setup (experiment_config.py) ------------------
metrics = [
	corpus_metrics.chi_square_distance,
	corpus_metrics.zipf_distance,
	corpus_metrics.classifier_distance,
	corpus_metrics.IRPR_distance,
	corpus_metrics.fid_distance,
	corpus_metrics.pr_distance,
	corpus_metrics.dc_distance,
	corpus_metrics.mauve_distance,
	corpus_metrics.traditional_biber_distance,
	corpus_metrics.zero_wasserstein_distance
]

metrics_names = [((str(dist).split()[1]).split('_')[0]).upper() for dist in metrics]

# Helper function for getting metric-dependent data.
def get_metric_dependant_data(metric, corpus: Corpus):
	if metric in (corpus_metrics.zipf_distance, corpus_metrics.chi_square_distance):
		c = STTokenizerEmbedder().tokenize_sentences(corpus)
	elif metric == corpus_metrics.zero_biber_distance:
		c = corpus
	else:
		c = STTokenizerEmbedder().embed_sentences(corpus)
	return c

# Helper function for getting all metric data.
def get_data_for_compcor_metrics(corpus):
	tokens = STTokenizerEmbedder(embedding_model_name = "all-MiniLM-L12-v2").tokenize_sentences(corpus)
	embeddings = STTokenizerEmbedder(embedding_model_name = "all-MiniLM-L12-v2").embed_sentences(corpus)
	return tokens, embeddings

def get_distances_from_compare_corpora(setA, setB):    
    tokensA, embeddingsA = get_data_for_compcor_metrics(setA)
    tokensB, embeddingsB = get_data_for_compcor_metrics(setB)
    distances = {}
    for metric_name, metric in zip(metrics_names, metrics):
        if metric in (corpus_metrics.zipf_distance, corpus_metrics.chi_square_distance):
            tempA, tempB = tokensA, tokensB
        elif metric in (corpus_metrics.traditional_biber_distance, corpus_metrics.zero_wasserstein_distance):
            tempA, tempB = setA, setB
        else:
            tempA, tempB = embeddingsA, embeddingsB
        distances[metric_name] = metric(corpus1=tempA, corpus2=tempB)

    return distances


In [4]:
# subject the real data to the same processing as the generated data
def clean_note(text):
    text = re.sub(r'^[\s"]+|[\s"]+$', '', text)   # strip edge quotes/spaces
    text = re.sub(r'\*', '', text)                 # remove asterisks
    text = re.sub(r'\s+', ' ', text)              # normalize whitespace
    return text.strip()

real_dataset = pd.read_csv(f'../realNotes/makeOneBigFile/dataRealAll.csv', index_col=None)

real_dataset = real_dataset.dropna(subset='Note')
real_dataset['Note'] = [clean_note(text) for text in real_dataset['Note'].tolist()]


In [5]:
input_generated_data_dirs = ['dataGeneration/syntheticNotesLocal/fakeNoteGeneration/processedFakeNoteSyntheticNotes', 'dataGeneration/syntheticNotesOnline/gpt3Notes', 'dataGeneration/syntheticNotesOnline/gpt4Notes']
all_dfs = {}
for directory in input_generated_data_dirs:
    temp_dfs = []
    for file in os.listdir(f'./{directory}'):
        if not os.path.isdir(f'./{directory}/{file}'):
            temp_dfs.append(pd.read_csv(f'./{directory}/{file}'))
    all_dfs[directory.split('/')[-1]] = pd.concat(temp_dfs)

In [6]:
dataset_metrics_averaged = {}

for dataset, temp_df in all_dfs.items():
    temp_results = defaultdict(list)
    
    for i in range(3):
        real_sample = real_dataset.sample(frac=1, random_state = i)['Note'].tolist()[:100]
        model_sample = temp_df.sample(frac=1, random_state = i)['report'].dropna().tolist()[:100]
        temp_metrics = get_distances_from_compare_corpora(real_sample, model_sample)

        for metric, value in temp_metrics.items():
            temp_results[metric].append(value)

    dataset_metrics_averaged[dataset] = {}                                       
    for metric, values in temp_results.items():
        dataset_metrics_averaged[dataset][metric] = sum(values) / len(values)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

The `tokenize` method is deprecated, please use `preprocess` instead.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Num real: 100 Num fake: 100
Num real: 100 Num fake: 100


WARNING clustering 200 points to 10 centroids: please provide at least 390 training points
[INFO] Using TTR for f_43_type_token
[INFO] All features normalized per 1000 tokens except: f_43_type_token and f_44_mean_word_length
INFO:pybiber.biber_analyzer:Zero-variance features retained (neutral scaling) in projection: ['f_07_second_person_pronouns', 'f_12_proverb_do', 'f_30_that_obj', 'f_32_wh_obj', 'f_35_because', 'f_37_if', 'f_47_hedges', 'f_59_contractions', 'f_60_that_deletion']


In [7]:
dataset = 'realToReal'
temp_results = defaultdict(list)

for i in range(3):
    real_sample = real_dataset.sample(frac=1, random_state = i)['Note'].tolist()
    assert len(real_sample) >= 200, "real samples must be over 200 samples long to prevent bias"
    real_start = real_sample[:100]
    real_end = real_sample[-100:]
    temp_metrics = get_distances_from_compare_corpora(real_start, real_end)

    for metric, value in temp_metrics.items():
        temp_results[metric].append(value)

dataset_metrics_averaged[dataset] = {}                                       
for metric, values in temp_results.items():
    dataset_metrics_averaged[dataset][metric] = sum(values) / len(values)

dataset = 'realToReal_2'
temp_results = defaultdict(list)

for i in range(3):
    real_sample = real_dataset.sample(frac=1, random_state = i)['Note'].tolist()
    assert len(real_sample) >= 200, "real samples must be over 200 samples long to prevent bias"
    real_start = real_sample[:100]
    real_end = real_sample[100:200]
    temp_metrics = get_distances_from_compare_corpora(real_start, real_end)

    for metric, value in temp_metrics.items():
        temp_results[metric].append(value)

dataset_metrics_averaged[dataset] = {}                                       
for metric, values in temp_results.items():
    dataset_metrics_averaged[dataset][metric] = sum(values) / len(values)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Num real: 100 Num fake: 100
Num real: 100 Num fake: 100


...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Num real: 100 Num fake: 100
Num real: 100 Num fake: 100


WARNING clustering 200 points to 10 centroids: please provide at least 390 training points
[INFO] Using TTR for f_43_type_token
[INFO] All features normalized per 1000 tokens except: f_43_type_token and f_44_mean_word_length
INFO:pybiber.biber_analyzer:Zero-variance features retained (neutral scaling) in projection: ['f_07_second_person_pronouns', 'f_10_demonstrative_pronoun', 'f_12_proverb_do', 'f_20_existential_there', 'f_23_wh_clause', 'f_29_that_subj', 'f_30_that_obj', 'f_32_wh_obj', 'f_33_pied_piping', 'f_35_because', 'f_36_though', 'f_47_hedges', 'f_50_discourse_particles', 'f_59_contractions', 'f_60_that_deletion', 'f_63_split_auxiliary']


In [8]:

df = pd.DataFrame.from_dict(dataset_metrics_averaged, orient='index')
df.reset_index(inplace=True)
df.rename(columns={'index': 'dataset'}, inplace=True)

df.to_csv("./realFakemetrics.csv", index=False)

In [9]:
df

,dataset,CHI,ZIPF,CLASSIFIER,IRPR,FID,PR,DC,MAUVE,TRADITIONAL,ZERO
0,processedFakeNoteSyntheticNotes,0.000000,0.186846,0.975203,0.345990,0.909971,0.797329,0.959554,0.989145,0.554923,0.255705
1,gpt3Notes,0.000000,0.136416,0.950653,0.326523,0.828887,0.750784,0.949262,0.964472,0.500461,0.250659
2,gpt4Notes,0.000000,0.174486,0.975997,0.345340,0.916107,0.771447,0.960601,0.986315,0.521236,0.278961
3,realToReal,0.667949,0.027829,0.452498,0.193086,0.241809,0.117652,0.067015,0.030778,0.191097,0.091320
4,realToReal_2,1.000000,0.033036,0.400513,0.187757,0.231786,0.111260,0.056596,0.028430,0.171535,0.089663
